# 2008 SELEC2007 per-machine L-H threshold scaling

Per-machine (AUG, JET, CMOD) log-linear OLS fits of `P_loss` on `(ne, Bt, Ip)` and `(ne, Bpol)`, with the `S` (separatrix area) exponent fixed to 1 in both. Data loaded via SimDB metadata only.

In [1]:
import numpy as np

import _simdb_common as sc

db = sc.get_db()

MACHINES = ["AUG", "JET", "CMOD"]
MU0 = 4 * np.pi * 1e-7

In [2]:
def load_machine(machine: str) -> np.ndarray:
    """SELEC2007-flagged 2008 slices for one machine: bt, ne, ip, pl (PLTH-preferred-else-PL), s, rgeo."""
    sims = sc.query_selected(db, "2008", "SELEC2007", machine=machine)
    rows = []
    for sim in sims:
        md = sim.meta_dict()
        selec = sc.temp(md, "SELEC2007", n=0)
        n = len(selec)
        if n == 0:
            continue
        bt = sc.path(md, "global_quantities", "b0", "value", n=n)
        ne = sc.path(md, "line_average", "n_e", "value", n=n)
        ip = sc.path(md, "global_quantities", "ip", "value", n=n)
        plth = sc.path(md, "global_quantities", "power_loss", "value", n=n)
        pl = sc.temp(md, "PL", n=n)
        s = sc.temp(md, "SPLASMA", "area_of_separatrix", n=n)
        rgeo = sc.path(md, "boundary", "geometric_axis_r", "value", n=n)

        n_slices = min(len(a) for a in (selec, bt, ne, ip, plth, pl, s, rgeo))
        sel_mask = selec[:n_slices] == 1
        if not sel_mask.any():
            continue
        pl_eff = np.where(np.isnan(plth[:n_slices]), pl[:n_slices], plth[:n_slices])
        for i in np.nonzero(sel_mask)[0]:
            rows.append((bt[i], ne[i], ip[i], pl_eff[i], s[i], rgeo[i]))

    dtype = [("bt", float), ("ne", float), ("ip", float), ("pl", float), ("s", float), ("rgeo", float)]
    return np.array(rows, dtype=dtype)


def valid_mask(d: np.ndarray) -> np.ndarray:
    """Finite & positive check over every core variable used by either model."""
    return (
        np.isfinite(d["bt"]) & np.isfinite(d["ne"]) & np.isfinite(d["ip"])
        & np.isfinite(d["pl"]) & np.isfinite(d["s"]) & np.isfinite(d["rgeo"])
        & (d["pl"] > 0) & (d["ne"] > 0) & (d["s"] > 0) & (d["rgeo"] > 0) & (d["ip"] != 0)
    )

In [3]:
def fit_loglinear(label, y, **regressors):
    """Log-linear OLS: y ~ 1 + sum_k log(regressors[k]). Prints C, exponent +/- SE, RMSE, R^2."""
    names = list(regressors)
    n = len(y)
    X = np.column_stack([np.ones(n)] + [np.log(regressors[k]) for k in names])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    p = X.shape[1]
    sigma2 = np.sum(resid**2) / max(n - p, 1)
    se = np.sqrt(np.diag(sigma2 * np.linalg.pinv(X.T @ X)))
    r2 = 1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2)
    rmse = np.sqrt(np.mean(resid**2))

    print(f"{label}: N={n}")
    print(f"  C (multiplying factor) = {np.exp(coef[0]):.4g}   [offset = {coef[0]:+.3f} +/- {se[0]:.3f}]")
    for j, name in enumerate(names, start=1):
        print(f"  {name} exponent = {coef[j]:+.3f} +/- {se[j]:.3f}")
    print(f"  RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}")
    return coef, se, n, r2, rmse

In [4]:
data = {}
for machine in MACHINES:
    d = load_machine(machine)
    d = d[valid_mask(d)]
    data[machine] = d
    print(f"{machine}: N={len(d)}")

AUG: N=175
JET: N=562
CMOD: N=115


## Model 1: (ne, Bt, Ip), S fixed to 1

In [5]:
results_model1 = {}
for machine in MACHINES:
    d = data[machine]
    y = np.log(d["pl"] / 1e6 / d["s"])
    results_model1[machine] = fit_loglinear(
        machine, y, ne=d["ne"] / 1e20, Bt=np.abs(d["bt"]), Ip=np.abs(d["ip"]) / 1e6,
    )

AUG: N=175
  C (multiplying factor) = 0.02939   [offset = -3.527 +/- 0.112]
  ne exponent = +0.418 +/- 0.072
  Bt exponent = +0.825 +/- 0.118
  Ip exponent = +0.065 +/- 0.141
  RMSE(log) = 0.225   R^2 = 0.514
JET: N=562
  C (multiplying factor) = 0.04601   [offset = -3.079 +/- 0.109]
  ne exponent = +0.780 +/- 0.061
  Bt exponent = +2.167 +/- 0.137
  Ip exponent = -1.585 +/- 0.132
  RMSE(log) = 0.319   R^2 = 0.610
CMOD: N=115
  C (multiplying factor) = 0.1003   [offset = -2.300 +/- 0.315]
  ne exponent = +0.586 +/- 0.079
  Bt exponent = +0.429 +/- 0.177
  Ip exponent = +0.745 +/- 0.154
  RMSE(log) = 0.219   R^2 = 0.574


## Model 2: (ne, Bpol), S fixed to 1

In [6]:
results_model2 = {}
for machine in MACHINES:
    d = data[machine]
    bpol = 2 * np.pi * MU0 * np.abs(d["ip"]) * d["rgeo"] / d["s"]
    y = np.log(d["pl"] / 1e6 / d["s"])
    results_model2[machine] = fit_loglinear(machine, y, ne=d["ne"] / 1e20, Bpol=bpol)

AUG: N=175
  C (multiplying factor) = 0.1327   [offset = -2.019 +/- 0.150]
  ne exponent = +0.381 +/- 0.082
  Bpol exponent = +0.678 +/- 0.126
  RMSE(log) = 0.257   R^2 = 0.368
JET: N=562
  C (multiplying factor) = 0.1873   [offset = -1.675 +/- 0.090]
  ne exponent = +0.935 +/- 0.069
  Bpol exponent = +0.608 +/- 0.057
  RMSE(log) = 0.366   R^2 = 0.489
CMOD: N=115
  C (multiplying factor) = 0.2931   [offset = -1.227 +/- 0.087]
  ne exponent = +0.589 +/- 0.081
  Bpol exponent = +0.963 +/- 0.140
  RMSE(log) = 0.226   R^2 = 0.550


In [7]:
def _fmt(coef, se, idx):
    return f"{coef[idx]:+.3f}+/-{se[idx]:.3f}"


print(f"{'machine':8s} {'model':6s} {'ne':>16s} {'Bt/Bpol':>16s} {'Ip':>16s} {'R^2':>6s} {'N':>4s}")
for machine in MACHINES:
    coef, se, n, r2, rmse = results_model1[machine]
    print(f"{machine:8s} {'M1':6s} {_fmt(coef, se, 1):>16s} {_fmt(coef, se, 2):>16s} {_fmt(coef, se, 3):>16s} {r2:6.3f} {n:4d}")
for machine in MACHINES:
    coef, se, n, r2, rmse = results_model2[machine]
    print(f"{machine:8s} {'M2':6s} {_fmt(coef, se, 1):>16s} {_fmt(coef, se, 2):>16s} {'--':>16s} {r2:6.3f} {n:4d}")

machine  model                ne          Bt/Bpol               Ip    R^2    N
AUG      M1       +0.418+/-0.072   +0.825+/-0.118   +0.065+/-0.141  0.514  175
JET      M1       +0.780+/-0.061   +2.167+/-0.137   -1.585+/-0.132  0.610  562
CMOD     M1       +0.586+/-0.079   +0.429+/-0.177   +0.745+/-0.154  0.574  115
AUG      M2       +0.381+/-0.082   +0.678+/-0.126               --  0.368  175
JET      M2       +0.935+/-0.069   +0.608+/-0.057               --  0.489  562
CMOD     M2       +0.589+/-0.081   +0.963+/-0.140               --  0.550  115


## Pooled fit: all 6 SELEC2007 machines, (ne, Bt, S), S free

In [8]:
POOLED_MACHINES = ["AUG", "CMOD", "D3D", "JET", "JFT2M", "JT60U"]
pooled = np.concatenate([load_machine(m) for m in POOLED_MACHINES])
pooled = pooled[valid_mask(pooled)]

y_pooled = np.log(pooled["pl"] / 1e6)
fit_loglinear("Pooled (6 machines)", y_pooled, ne=pooled["ne"] / 1e20, Bt=np.abs(pooled["bt"]), S=pooled["s"])

Pooled (6 machines): N=1024
  C (multiplying factor) = 0.04878   [offset = -3.020 +/- 0.057]
  ne exponent = +0.717 +/- 0.035
  Bt exponent = +0.803 +/- 0.032
  S exponent = +0.941 +/- 0.019
  RMSE(log) = 0.307   R^2 = 0.830


(array([-3.02035672,  0.71731826,  0.80318175,  0.94116125]),
 array([0.05746237, 0.03534841, 0.03218542, 0.01905978]),
 1024,
 np.float64(0.8302438189804329),
 np.float64(0.30708712918570275))